In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# The heat_equation_residual function is no longer needed in its original form
# and its logic will be integrated into the loss_function.
# def heat_equation_residual(u, x, t, alpha):
#  u_t = tf.gradients(u, t)[0]
#  u_xx = tf.gradients(tf.gradients(u, x)[0], x)[0]
#  return u_t - alpha * u_xx

def create_pinn_model():
    model = keras.Sequential()
    model.add(layers.Input(shape=(2,))) # Input: [x, t]
    model.add(layers.Dense(50, activation='tanh'))
    model.add(layers.Dense(50, activation='tanh'))
    model.add(layers.Dense(1)) # Output: u(x, t)
    return model

# The loss_function's logic for calculating derivatives is now integrated into train_pinn
# def loss_function(model, x, t, alpha):
#     x = tf.cast(x, tf.float32)
#     t = tf.cast(t, tf.float32)
#     with tf.GradientTape(persistent=True) as tape:
#         tape.watch(x)
#         tape.watch(t)
#         u_pred = model(tf.concat([x, t], axis=1))
#         u_t = tape.gradient(u_pred, t)
#         u_x = tape.gradient(u_pred, x)
#         u_xx = tape.gradient(u_x, x)
#     del tape
#     residual = u_t - alpha * u_xx
#     return tf.reduce_mean(tf.square(residual))

def train_pinn(model, x_train, t_train, alpha, epochs):
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    for epoch in range(epochs):
        with tf.GradientTape(persistent=True) as tape: # Persistent tape for higher-order derivatives
            # Ensure x and t are float32 for consistency with model and gradient operations
            x_train_f = tf.cast(x_train, tf.float32)
            t_train_f = tf.cast(t_train, tf.float32)

            # Watch the input variables x and t to compute gradients with respect to them
            # These are external inputs, so we watch them to compute their derivatives
            tape.watch(x_train_f)
            tape.watch(t_train_f)

            # Predict u(x, t) using the model
            # This operation, and thus the model's trainable variables, are now within this tape's scope
            u_pred = model(tf.concat([x_train_f, t_train_f], axis=1))

            # Compute first-order derivatives
            u_t = tape.gradient(u_pred, t_train_f)
            u_x = tape.gradient(u_pred, x_train_f)

            # Compute second-order derivative u_xx = d(u_x)/d(x)
            u_xx = tape.gradient(u_x, x_train_f)

            # Calculate the residual of the heat equation
            # If any of the gradients are None, it indicates a problem in graph construction
            # or inputs not being watched. For now, assume they are not None.
            residual = u_t - alpha * u_xx

            # Calculate the loss
            loss_value = tf.reduce_mean(tf.square(residual))

        # Compute gradients of the loss with respect to the model's trainable variables
        # The model's variables were involved in computing u_pred, which led to loss_value,
        # all within the same tape. So, this should now correctly trace.
        grads = tape.gradient(loss_value, model.trainable_variables)

        # Filter out None gradients and replace them with zeros
        # This prevents the optimizer from crashing if some gradients are None
        filtered_grads = [grad if grad is not None else tf.zeros_like(var)
                          for grad, var in zip(grads, model.trainable_variables)]

        optimizer.apply_gradients(zip(filtered_grads, model.trainable_variables))

        # Delete the persistent tape to release resources
        del tape

        if epoch % 100 == 0:
            print(f"Epoch: {epoch}, Loss: {loss_value.numpy()}")

def generate_data(N):
    x = np.random.rand(N, 1) # Spatial domain
    t = np.random.rand(N, 1) # Temporal domain
    # Convert to TensorFlow tensors and ensure float32 type
    return tf.convert_to_tensor(x, dtype=tf.float32), tf.convert_to_tensor(t, dtype=tf.float32)

# Parameters
alpha = 0.01
N_train = 1000
epochs = 10000

# Generate data
x_train, t_train = generate_data(N_train)

# Create and train the model
model = create_pinn_model()
train_pinn(model, x_train, t_train, alpha, epochs)

# Evaluate the model after training (optional)


/usr/local/lib/python3.13/dist-packages/keras/src/optimizers/base_optimizer.py:870: UserWarning: Gradients do not exist for variables ['sequential_2/dense_8/bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


Epoch: 0, Loss: 0.006292775738984346
Epoch: 100, Loss: 1.4725917026225943e-05
Epoch: 200, Loss: 8.107274879876059e-06
Epoch: 300, Loss: 4.744672423839802e-06
Epoch: 400, Loss: 3.2380391985498136e-06
Epoch: 500, Loss: 2.511925004000659e-06
Epoch: 600, Loss: 2.046955842160969e-06
Epoch: 700, Loss: 1.6691978999006096e-06
Epoch: 800, Loss: 1.3388329307417735e-06
Epoch: 900, Loss: 1.0506601029192097e-06
Epoch: 1000, Loss: 8.046527568694728e-07
Epoch: 1100, Loss: 6.001435508551367e-07
Epoch: 1200, Loss: 4.350985420842335e-07
Epoch: 1300, Loss: 3.062245639284811e-07
Epoch: 1400, Loss: 2.0919094367854996e-07
Epoch: 1500, Loss: 1.3897326311962388e-07
Epoch: 1600, Loss: 9.025922764749339e-08
Epoch: 1700, Loss: 5.7910984452291814e-08
Epoch: 1800, Loss: 3.736468556780892e-08
Epoch: 1900, Loss: 1.6180248962882615e-07
Epoch: 2000, Loss: 2.525598041813737e-08
Epoch: 2100, Loss: 1.877942779060504e-08
Epoch: 2200, Loss: 2.763503914593457e-07
Epoch: 2300, Loss: 1.5789083818162908e-08
Epoch: 2400, Loss: 